# 开盘交互固定训练与回测
两组均含四个开盘字段，候选增加两个预登记乘积项。6次拟合、6套独立成本账户；复用历史只作诊断。

In [1]:
from pathlib import Path
import os,json,qlib
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/opening_interaction_training_20261015'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us')
from src.workflow.opening_state_training import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
print(summary)

# 开盘交互训练与回测：不晋升

本轮实际完成6次新拟合（两组、各3个固定seed、300轮末轮等权）及6套Qlib原生完整账户。两组同用2018-10-12至2022-12-21的1056条成熟主训练样本及20741条辅助样本，标签成熟截止2023-01-01。对照含原四个开盘字段；候选仅增加SOXL、SOXX各一个“隔夜gap × 前15分钟收益”交互项。初始输出相同，原输入缩放器相同，交互仅在训练期拟合缩放，缺失保持未知。重训对照三种成本收益与原四字段模型一致。

这检验线性配置头是否缺少条件表达能力，没有引入新的独立市场信息。决策09:46，只用已结束09:45及以前行情；成交09:47。训练与评价时序分开，但评价历史已多次使用，属于复用历史诊断，不是全新独立测试集。

回测2023-01-03信号至2026-10-09退出，共944区间；本金10万美元，沿用已冻结IBKR Pro Tiered费用投影，0/2/5bp账户独立结算。该费表是固定研究投影，不声称为各历史日期实际账单。

|账户|0bp净收益|2bp净收益|5bp净收益|
|---|---:|---:|---:|
|四字段对照|102.24%|139.55%|118.44%|
|增加交互|101.92%|136.00%|116.34%|
|SOXX|387.75%|387.73%|387.70%|

2bp候选监控最大回撤31.20%，超过30%门槛；126日滚动正超额占比24.79%。2023/2024/2025/2026年候选收益35.87%/31.05%/-5.90%/40.85%，同期SOXX56.60%/20.97%/46.80%/75.38%（2026为不完整年度）。仅2024跑赢。三个成本下候选均弱于配对对照，对照差值的区块区间均跨零，不能声称交互因果上必然有害；但无证据支持改进。CVaR门通过，净超额、回撤、年度、滚动、区块置信门均失败。

决策not_promoted_opening_interaction，生产未替换，稳定超额目标未完成。关闭这组预登记交互，不改符号、窗口、seed、epoch或风险阈值挽救。下一项需训练期OOF证据识别上涨阶段现金配置的独立预测信息，或新的可用信息来源；不能把原字段再组合后宣称已找到alpha，也不能靠减少现金迎合已看过的测试行情。

Qlib模型、预测、逐日账户与来源存于Recorder/NAS；NAS冷恢复逐对象哈希通过。既有开盘时点及配置头测试2项通过，初始化等价、同训练样本、同缩放和完整账户对账在notebook执行中检查。全分钟最大回撤、真实派息及实际成交时延仍未认证。
